# Map of Italian Science — Organisation-Level Citation Analysis

#### Author: Peiz Yu

## Research questions addressed in this notebook

- **Which external organisations cite the publications of each Italian institution** recorded in OpenCitations (incoming citations), and at what volume?
- **Which external organisations are cited by each Italian institution** (outgoing citations), and at what volume?
- **How symmetric or asymmetric** is the citation exchange between each Italian institution and its major partners — and does this vary systematically across institutions?
- **How do international and domestic citation relationships differ** — and what does the domestic inter-institutional layer within Italy look like at the organisation level?

The six Italian institutions examined are: University of Bologna (UNIBO), University of Milan (UNIMI), University of Padua (UNIPD), University of Turin (UNITO), University of Eastern Piedmont (UPO), and Scuola Normale Superiore (SNS).

*This notebook is the organisation-level companion to the country-level analysis. It follows the same data pipeline and, where relevant, the same exclusion logic — operating at partner-institution granularity rather than country granularity.*

*All visualisations support two analytical modes, toggled interactively within each chart:*
- ***Exclude Italian partners*** *— mirrors the scope of the country-level analysis, isolating international citation flows and enabling direct cross-notebook comparison.*
- ***Include Italian partners*** *— reveals the domestic inter-institutional citation network within Italy, a layer invisible at the country level, where relationships between Italian universities and research institutions emerge as structurally significant.*

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import pycountry
from plotly.subplots import make_subplots
from pathlib import Path
import warnings
import sys
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.0f}'.format)

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("⚠ ipywidgets not available — set EXCLUDE_ITALIAN_PARTNERS manually in the config cell.")


## Configuration

In [ ]:
# ── Paths ──
# If __file__ exists, use it. Otherwise, use the Current Working Directory.
try:
    CURRENT_DIR = Path(__file__).resolve().parent
except NameError:
    CURRENT_DIR = Path.cwd()

# Add parent directory of data_viz to sys.path to allow importing from src
if CURRENT_DIR.name == "data_viz":
    sys.path.append(str(CURRENT_DIR.parent))
else:
    sys.path.append(str(CURRENT_DIR))

import src.data_utils as du
from src.data_utils import * 

from src.validation import * 

# ── Direction colours (consistent with country-level notebook) ──
COLORS = {"incoming": "#B7990D", "outgoing": "#320E3B"}

TOP_N     = 15
RECIP_TOP = 500

COUNTRY_COLORS = {
    "United States":    "#1f77b4",
    "France":           "#B7990D",
    "United Kingdom":   "#d62728",
    "Germany":          "#2ca02c",
    "China":            "#ff7f0e",
    "Spain":            "#9467bd",
    "Japan":            "#e377c2",
    "Canada":           "#17becf",
    "Australia":        "#bcbd22",
    "The Netherlands":  "#8c564b",
    "Switzerland":      "#aec7e8",
    "Russia":           "#c5b0d5",
    "India":            "#ffbb78",
    "South Korea":      "#98df8a",
    "Brazil":           "#ff9896",
    "Poland":           "#f7b6d2",
    "Belgium":          "#c49c94",
    "Türkiye":          "#dbdb8d",
    "Sweden":           "#9edae5",
    "Finland":          "#393b79",
    "Denmark":          "#637939",
    "Taiwan":           "#8c6d31",
    "Greece":           "#843c39",
    "Portugal":         "#7b4173",
    "Austria":          "#5254a3",
    "Italy":            "#e6550d",
}
OTHER_COUNTRY_COLOR = "#cccccc"

LEGEND_COUNTRIES = [
    "France","United States","United Kingdom","Germany","Spain","China",
    "Italy","Canada","Switzerland","Japan","Denmark","Brazil","Russia",
    "Australia","Greece","Poland","The Netherlands","Finland","Sweden",
    "India","South Korea","Belgium",
]

# ── Italian partner exclusion ─────────────────────────────────────────────
# Controls whether Italian partner institutions (other than the focal
# institution itself) are included or excluded from the analysis.
#   False → include Italian partners (reveals domestic inter-institutional ties)
#   True  → exclude Italian partners (mirrors country-level analysis scope)
# This flag is toggled interactively by the widget in Section 1.
EXCLUDE_ITALIAN_PARTNERS = False
du.EXCLUDE_ITALIAN_PARTNERS = EXCLUDE_ITALIAN_PARTNERS

# ── Institution colour palette (consistent with country-level notebook) ────
INST_COLORS = {
    "UNIBO": "#264653",
    "UNIMI": "#2a9d8f",
    "UNIPD": "#8ab17d",
    "UNITO": "#e9c46a",
    "UPO":   "#f4a261",
    "SNS":   "#e76f51",
}

# Alias for compatibility with country-level notebook variable name
DIR_COLORS = COLORS


## 1. Setup & Data Loading

### Data cleaning and standardisation

The raw CSV files contain known inconsistencies in country naming (e.g. `"China (People's Republic of)"` and `"China"` refer to the same country). Before any aggregation, all country names are mapped to a canonical form via `COUNTRY_NAME_MAP`.

Each institution's data also contains a row for the focal institution itself (self-citations in the OpenCitations graph). Following the same logic as the country-level analysis — which excludes the focal country — we exclude **only the focal institution's own row**, retaining all other Italian institutions. This allows domestic inter-institutional citation relationships to surface in the analysis.

### Data loading and transformation pipeline

The following functions implement the data ingestion process:

- **`normalise_country_names(df)`** — applies the canonical name map to the `country_name` column; called immediately after `read_csv` before any other operation.
- **`load_org_data(institution)`** — reads the two CSV files for one institution (incoming and outgoing), normalises country names, removes the self-citation row, and tags each row with the institution key. Returns `(incoming_df, outgoing_df)`.
- **`load_all_available()`** — iterates over all six institutions, calls `load_org_data` for each, and collects results into a single `ALL_DATASETS` dictionary. Institutions whose CSV files are not yet present are skipped with a printed warning, so the notebook runs on partial data without error.


### Data cleaning and standardisation

The following discovery cell scans all CSV files across the six institutions in both directions to identify:
- organisations whose `country_name` varies for the same `country_code` (country-level variant, same issue as in the country notebook)
- duplicate `legal_name` / `ror` rows that need to be aggregated before analysis

Based on these findings we apply a canonical `COUNTRY_NAMES` map (keyed on ISO country codes, identical to the country-level notebook) and aggregate counts for any rows that share the same `(ror, country_code)` pair after normalisation.


#### Data Loading and Transformation Pipeline

The following functions modularise the data ingestion process, mirroring the country-level pipeline:
- `load_org_data` — standardises country names, drops rows with missing identifiers, removes the focal institution's self-citation row, aggregates counts for any `(ror, country_code)` duplicates that arise after normalisation, and tags each row with `institution` and `direction` metadata
- `load_all_available` — iterates over all six institutions, calls `load_org_data` for each direction, and collects results into a shared `ALL_DATASETS` dictionary; institutions with missing files are skipped gracefully


In [ ]:

ALL_DATASETS = load_all_available()

if "UNIBO" in ALL_DATASETS:
    incoming_df, outgoing_df = ALL_DATASETS["UNIBO"]
else:
    print("⚠ UNIBO data not loaded — check BASE_PATH in the config cell.")


### Optional Data Validation

The following utilities are not required for the analysis itself. They are provided for quality-control purposes and should only be executed when new datasets are received or when country normalization rules are updated.

- `validate_dataset()` checks for duplicate country codes and missing values.
- `export_cleaned_csvs()` generates cleaned versions of the original datasets after normalization and aggregation.

In [ ]:
for inst in INSTITUTIONS:
    for direction in ["incoming", "outgoing"]:
        print(f"\n{inst} {direction}")
        validate_dataset(
            load_org_data(inst, direction),
            duplicate_subset=["ror","country_code"]
        )

export_cleaned_csvs(
    loader_function=load_org_data,
    filename_prefix="organizations",     
    output_dir=BASE_PATH.parent / "visualizations",
    institutions=INSTITUTIONS
)

### Italian Partner Filter

Use the toggle below to include or exclude Italian partner institutions from the analysis.
Switching the toggle automatically reloads all datasets — re-run the visualisation cells afterwards to update the charts.

> **Include Italian partners** — reveals domestic inter-institutional citation ties (e.g. UNIBO↔UNIPD bilateral relationship, UNIMI's clinical network via IRCCS).

> **Exclude Italian partners** — mirrors the scope of the country-level analysis, focusing on international relationships only.


In [ ]:
import src.data_utils as du 
if HAS_WIDGETS:
    toggle = widgets.ToggleButtons(
        options=["Include Italian partners", "Exclude Italian partners"],
        value="Include Italian partners" if not EXCLUDE_ITALIAN_PARTNERS
              else "Exclude Italian partners",
        description="",
        style={"button_width": "220px"},
        button_style="",
    )
    output_log = widgets.Output()

    def on_toggle_change(change):
        global EXCLUDE_ITALIAN_PARTNERS, ALL_DATASETS, incoming_df, outgoing_df
        EXCLUDE_ITALIAN_PARTNERS = (change["new"] == "Exclude Italian partners")
        du.EXCLUDE_ITALIAN_PARTNERS = EXCLUDE_ITALIAN_PARTNERS
        with output_log:
            clear_output(wait=True)
            print(f"Reloading data — Italian partners: "
                  f"{'excluded' if EXCLUDE_ITALIAN_PARTNERS else 'included'} ...")
            ALL_DATASETS = load_all_available()
            if "UNIBO" in ALL_DATASETS:
                incoming_df, outgoing_df = ALL_DATASETS["UNIBO"]
            print("✓ Done. Re-run the visualisation cells to update the charts.")

    toggle.observe(on_toggle_change, names="value")
    display(toggle, output_log)

else:
    # Fallback: set manually
    # EXCLUDE_ITALIAN_PARTNERS = False   ← include Italian partners
    # EXCLUDE_ITALIAN_PARTNERS = True    ← exclude Italian partners
    print(f"Italian partners currently: "
          f"{'excluded' if EXCLUDE_ITALIAN_PARTNERS else 'included'}")
    print("To change: set EXCLUDE_ITALIAN_PARTNERS in the config cell and re-run from there.")


## 2. Per-Institution Analysis

### 2a — Combined Butterfly Chart

Each subplot shows the **top-15 partner organisations** ranked by total citation volume (incoming + outgoing combined). Bars extending **left** (gold) = organisations that **cite the focal institution** (incoming). Bars extending **right** (purple) = organisations **cited by the focal institution** (outgoing). Other Italian institutions are included, so domestic relationships surface alongside international ones.


In [ ]:
def build_butterfly_data(inb, out, top_n=TOP_N):
    """Merge incoming/outgoing, pick top_n orgs by total, return long-form DataFrame."""
    inb_agg = inb.groupby(["legal_name","country_name","country_code"])["count"].sum().reset_index()
    out_agg = out.groupby(["legal_name","country_name","country_code"])["count"].sum().reset_index()

    merged = pd.merge(
        inb_agg.rename(columns={"count": "incoming"}),
        out_agg.rename(columns={"count": "outgoing"}),
        on=["legal_name","country_name","country_code"], how="outer",
    ).fillna(0)

    merged["total"] = merged["incoming"] + merged["outgoing"]
    top = merged.nlargest(top_n, "total").copy()

    # Y-axis label: truncated name + country
    top["label"] = top.apply(
        lambda r: (r["legal_name"][:36]+"…" if len(r["legal_name"]) > 38 else r["legal_name"])
                  + "  (" + r["country_name"] + ")",
        axis=1,
    )

    inb_long = top[["label","legal_name","country_name","incoming"]].copy()
    inb_long["direction"] = "incoming"
    inb_long["value"]     = -inb_long["incoming"]
    inb_long["count"]     =  inb_long["incoming"]
    inb_long = inb_long.drop(columns="incoming")

    out_long = top[["label","legal_name","country_name","outgoing"]].copy()
    out_long["direction"] = "outgoing"
    out_long["value"]     =  out_long["outgoing"]
    out_long["count"]     =  out_long["outgoing"]
    out_long = out_long.drop(columns="outgoing")

    long_df = pd.concat([inb_long, out_long], ignore_index=True)
    order = top.sort_values("total")["label"].tolist()
    long_df["label"] = pd.Categorical(long_df["label"], categories=order, ordered=True)
    return long_df.sort_values("label")


def plot_butterfly_combined(datasets: dict, top_n: int = TOP_N) -> go.Figure:
    """
    3 × 2 subplot grid — one butterfly chart per institution.
    Each subplot has its own x-axis scale.
    """
    institutions = list(datasets.keys())
    ncols, nrows = 2, 3

    fig = make_subplots(
        rows=nrows, cols=ncols,
        subplot_titles=[INSTITUTION_LABELS[inst] for inst in institutions],
        horizontal_spacing=0.26,
        vertical_spacing=0.055,
    )

    legend_added = set()

    for i, inst in enumerate(institutions):
        row, col = i // ncols + 1, i % ncols + 1
        inb, out = datasets[inst]
        long_df  = build_butterfly_data(inb, out, top_n)
        max_val  = long_df["count"].max()

        for direction, color in [("incoming", COLORS["incoming"]),
                                  ("outgoing",  COLORS["outgoing"])]:
            df_d = long_df[long_df["direction"] == direction]
            show = direction not in legend_added

            fig.add_trace(go.Bar(
                x=df_d["value"],
                y=df_d["label"],
                orientation="h",
                name=direction,
                marker_color=color,
                legendgroup=direction,
                showlegend=show,
                customdata=df_d[["count","direction","country_name","legal_name"]].values,
                hovertemplate=(
                    "<b>%{customdata[3]}</b><br>"
                    "%{customdata[2]}<br>"
                    "Citations: %{customdata[0]:,.0f}<br>"
                    "Direction: %{customdata[1]}<extra></extra>"
                ),
            ), row=row, col=col)

            if show:
                legend_added.add(direction)

        fig.update_xaxes(
            range=[-max_val * 1.12, max_val * 1.12],
            tickformat=",", row=row, col=col,
        )
        fig.add_vline(x=0, line_width=1, line_color="gray", row=row, col=col)

    fig.update_layout(
        template="plotly_white",
        height=nrows * 530,
        barmode="overlay",
        bargap=0.15,
        title_text=f"Top {top_n} Organisations — Incoming vs Outgoing Citations",
        title_x=0.5,
        legend=dict(
            orientation="h", yanchor="bottom", y=1.01,
            xanchor="center", x=0.5,
            title_text="",
        ),
    )
    fig.update_yaxes(tickfont=dict(size=8.5))
    return fig


plot_butterfly_combined(ALL_DATASETS, TOP_N).show()


#### Butterfly Chart — Findings: Including Italian Partners

**CNRS is the dominant partner at every institution.** It appears at the top of all six subplots with broadly balanced incoming and outgoing bars — the most universally symmetric high-volume relationship in the dataset.

**Domestic inter-institutional citations are substantial and visible.** Italian universities appear in each other's top-15 across all institutions. The most prominent shared partners are Sapienza University of Rome (in five of six top-15 lists) and University of Bologna and University of Padua (in four each). This confirms that the Italian national research network is a major citation ecosystem in its own right, not a secondary backdrop to international partnerships.

**UNIBO and UNIPD have a strong bilateral tie.** University of Padua ranks 5th for UNIBO and University of Bologna ranks 5th for UNIPD, both with roughly balanced bars — the most symmetric large Italian–Italian pair in the dataset.

**UPO is the most domestically concentrated institution.** Seven of its fifteen top partners are Italian, including Università degli Studi del Piemonte Orientale (predecessor institution, ranked 1st with the longest bars in the UPO subplot) and University of Turin (ranked 2nd). No other institution shows this degree of domestic top-15 concentration. CERN and IN2P3 are UPO's main international outgoing-skewed partners, with clearly longer purple bars.

**UNIMI's incoming bars are visibly longer than outgoing for IRCCS and CNR.** Italian hospital research networks (IRCCS) and the National Research Council cite UNIMI substantially more than UNIMI cites them back, reflecting UNIMI's role as a primary reference institution for Italian clinical and applied research. Other top partners (Harvard University Press, CSIC) show the typical outgoing bias seen across all institutions.

**SNS has the most symmetric top-15 and the strongest geographic proximity tie.** University of Pisa ranks 3rd for SNS (incoming ≈ outgoing, asymmetry −0.02) — a near-perfectly balanced partnership between two institutions in the same city. Harvard University Press is absent from SNS's top-15 entirely, replaced by CERN (+0.17) and IN2P3 (+0.03) as the dominant physics infrastructure partners.

**UNITO shows consistent outgoing bias for its largest international partners.** Harvard University Press and Texas Tech University System both show clearly longer outgoing bars, and CERN/IN2P3 appear similarly outgoing-skewed — consistent with UNITO citing physics and computational reference literature more than it is cited back by those institutions.


#### Butterfly Chart — Findings: Excluding Italian Partners

**CNRS remains the dominant partner at every institution.** Its position and near-symmetric bars are unchanged — confirming that this finding is fully independent of the Italian partner filter.

**Italian slots are replaced by international partners ranking 16th or lower in the inclusion mode.** The most notable replacements are institution-specific: California Baptist University and Sorbonne Université enter UNIBO's list; Imperial College London and additional French physics institutions enter SNS's and UPO's lists. The overall US–France–UK triad structure is reinforced once domestic partners no longer compete for top-15 rank.

**Harvard University Press's outgoing asymmetry becomes more visually prominent.** Without near-symmetric Italian partners pulling the subplot average toward the diagonal, the outgoing bars of Harvard University Press stand out more clearly in every chart. This makes the publisher-driven citation dependency more legible across all six institutions simultaneously.

**UNITO's outgoing bias is the clearest in this mode.** With Italian institutions removed, the dark purple bars of UNITO's international partners extend more uniformly further than the gold bars across the full subplot — making UNITO the most visibly asymmetric institution in the international-only view. The contrast with SNS's symmetric chart is starker here than in the inclusion mode.

**UPO's profile is transformed entirely.** The seven Italian partners that dominated its top-15 in the inclusion mode disappear. Physics infrastructure (CERN, IN2P3) and French, US, and Spanish research universities fill the list instead, revealing an international citation profile structurally similar to SNS.

**SNS's proximity tie with University of Pisa disappears.** University of Pisa is excluded as an Italian institution. SNS's top-15 becomes a purely international physics list, but the overall near-symmetry of the chart is maintained — SNS's international partners are also broadly balanced — so the institution remains the most symmetric in this mode as well.


#### Butterfly Chart — Comparing the Two Perspectives

**What is stable across both modes:** CNRS's dominance and near-symmetry, Harvard University Press's outgoing bias, SNS's overall chart symmetry (even without University of Pisa), and UNITO's outgoing skew for its largest international partners. These are structural features of the citation network, not artefacts of the Italian inclusion decision.

**What changes:** The most dramatic difference is at UPO, where the entire top-15 composition shifts — from a domestically concentrated list to an international physics profile — making it the institution most sensitive to the filter choice. UNIMI's distinctive IRCCS signal is only visible in the inclusion mode; in the exclusion mode, UNIMI's international mix is broadly comparable to UNIPD's.

**Which mode to use:** The inclusion mode is the appropriate choice when the research question concerns the full citation neighbourhood of each institution, including domestic inter-institutional ties. The exclusion mode is appropriate when the research question focuses on international citation flows and requires direct comparability with the country-level analysis.


### 2b — Reciprocity Scatter (Interactive)

Organisations appearing in **both** the top-500 incoming and top-500 outgoing lists for a given institution. Select an institution from the dropdown. The dashed diagonal marks perfect reciprocity — points above cite less back than they are cited; points below cite more than they receive. Bubble size = total citations · colour = country.


In [ ]:
import json

def compute_reciprocity(inb, out, recip_top=RECIP_TOP):
    inb_top = (inb.groupby(["legal_name","country_name"])["count"].sum()
               .reset_index().nlargest(recip_top,"count")
               .rename(columns={"count":"incoming"}))
    out_top = (out.groupby(["legal_name","country_name"])["count"].sum()
               .reset_index().nlargest(recip_top,"count")
               .rename(columns={"count":"outgoing"}))
    recip = pd.merge(inb_top[["legal_name","country_name","incoming"]],
                     out_top[["legal_name","country_name","outgoing"]],
                     on=["legal_name","country_name"], how="inner")
    recip["total"]     = recip["incoming"] + recip["outgoing"]
    recip["asymmetry"] = (recip["outgoing"] - recip["incoming"]) / recip["total"]
    return recip


def _title_text(label: str, recip_top: int) -> str:
    return (f"Reciprocity Scatter — Top-{recip_top} Bilateral Partners  ({label})<br>")


def build_reciprocity_figure(datasets: dict, recip_top: int = RECIP_TOP,
                             default: str | None = None):
    """
    Builds the figure WITH a dropdown (for notebook use) and returns it
    together with a {label: visibility-list} map. The HTML export strips
    the dropdown and replaces it with pill buttons driven by the same map.
    """
    institutions  = list(datasets.keys())
    legend_groups = LEGEND_COUNTRIES + ["Other"]
    n_groups      = len(legend_groups)
    block         = n_groups + 1                      # country traces + diagonal
    fig = go.Figure()

    for inst in institutions:
        inb, out = datasets[inst]
        recip    = compute_reciprocity(inb, out, recip_top)

        # size scaling computed on the FULL institution dataset,
        # so bubbles stay comparable across country traces
        t = recip["total"]
        recip = recip.assign(
            size=5 + 32 * (t - t.min()) / (t.max() - t.min() + 1),
            group=recip["country_name"].where(
                recip["country_name"].isin(LEGEND_COUNTRIES), "Other"
            ),
        )

        for country in legend_groups:
            sub = recip[recip["group"] == country]
            fig.add_trace(go.Scatter(
                x=sub["incoming"], y=sub["outgoing"],
                mode="markers",
                name=country,
                legendgroup=country,
                showlegend=False,
                marker=dict(
                    size=sub["size"],
                    color=COUNTRY_COLORS.get(country, OTHER_COUNTRY_COLOR),
                    opacity=0.72,
                    line=dict(width=0.5, color="white"),
                ),
                customdata=sub[["legal_name", "country_name",
                                "incoming", "outgoing", "total"]].values,
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>%{customdata[1]}<br>"
                    "Incoming: %{customdata[2]:,.0f}<br>"
                    "Outgoing: %{customdata[3]:,.0f}<br>"
                    "Total: %{customdata[4]:,.0f}<extra></extra>"
                ),
            ))

        ax_min = float(min(recip[["incoming", "outgoing"]].min()))
        ax_max = float(max(recip[["incoming", "outgoing"]].max()))
        fig.add_trace(go.Scatter(
            x=[ax_min, ax_max], y=[ax_min, ax_max],
            mode="lines", line=dict(color="gray", dash="dash", width=1),
            showlegend=False, hoverinfo="skip",
        ))

    n_inst_traces = len(institutions) * block

    # Country legend traces (always visible, clickable filters)
    for country in legend_groups:
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode="markers",
            name=country,
            marker=dict(color=COUNTRY_COLORS.get(country, OTHER_COUNTRY_COLOR), size=9),
            showlegend=True, visible=True,
            legendgroup=country,
        ))

    # Visibility map shared by the dropdown AND the HTML pill buttons
    vis_map = {}
    for i, inst in enumerate(institutions):
        vis = [False] * n_inst_traces + [True] * n_groups
        for j in range(block):
            vis[i * block + j] = True
        vis_map[INSTITUTION_LABELS[inst]] = vis

    if default is None:
        default = INSTITUTION_LABELS[institutions[0]]

    # Apply default selection to the figure itself
    default_vis = vis_map[default]
    for k, tr in enumerate(fig.data):
        tr.visible = default_vis[k]

    # Dropdown for the notebook, built from the same vis_map
    dropdown_buttons = [
        dict(label=label, method="update",
             args=[{"visible": vis},
                   {"title": {"text": _title_text(label, recip_top), "x": 0.5}}])
        for label, vis in vis_map.items()
    ]

    fig.update_layout(
        template="plotly_white",
        height=640,
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="#FFFFFF", family="Inter, sans-serif"),
        title=dict(
            text=_title_text(default, recip_top),
            font=dict(family='Playfair Display, serif', size=18),
            x=0.5,
        ),
        xaxis=dict(type="log", title="Incoming citations",
                   gridcolor="rgba(255,255,255,0.07)",
                   zerolinecolor="rgba(255,255,255,0.15)"),
        yaxis=dict(type="log", title="Outgoing citations",
                   gridcolor="rgba(255,255,255,0.07)",
                   zerolinecolor="rgba(255,255,255,0.15)"),
        legend_title_text="Country",
        legend=dict(itemclick="toggleothers", itemdoubleclick="toggle",
                    groupclick="togglegroup"),
        updatemenus=[dict(
            buttons=dropdown_buttons,
            direction="down",
            showactive=False,
            x=0.0, xanchor="left",
            y=1.13, yanchor="top",
            bordercolor="rgba(255, 255, 255, 0.2)",
            font=dict(size=12, color="#FFFFFF", family="Inter, sans-serif"),
        )],
        margin=dict(t=120),
    )

    return fig, vis_map


# Pill-style button bar (HTML export only)
BUTTONS_CSS = """
<style>
.recip-toolbar {
  display: flex; align-items: center; flex-wrap: wrap; gap: 8px;
  font-family: 'Inter', sans-serif;
  margin: 4px 0 12px 0;
}
.recip-toolbar .recip-label {
  color: rgba(255,255,255,0.45);
  font-size: 11px; font-weight: 600;
  letter-spacing: 0.14em; text-transform: uppercase;
  margin-right: 4px;
}
.recip-btn {
  appearance: none; cursor: pointer;
  background: transparent;
  color: rgba(255,255,255,0.85);
  border: 1px solid rgba(255,255,255,0.25);
  border-radius: 999px;
  padding: 5px 14px;
  font-family: 'Inter', sans-serif;
  font-size: 11px; font-weight: 600;
  letter-spacing: 0.08em; text-transform: uppercase;
  transition: background .15s ease, color .15s ease, border-color .15s ease;
}
.recip-btn:hover { border-color: rgba(255,255,255,0.55); }
.recip-btn.active {
  background: #B89B1E;            /* gold */
  border-color: #B89B1E;
  color: #2B1B3D;                 /* dark plum text */
}
</style>
"""

# Pill text: acronym for each institution label. Adjust to your actual
# INSTITUTION_LABELS values; anything missing falls back to initials.
INSTITUTION_ACRONYMS = {
    "University of Bologna":           "UNIBO",
    "University of Milan":             "UNIMI",
    "University of Padua":             "UNIPD",
    "University of Turin":             "UNITO",
    "University of Eastern Piedmont":  "UPO",
    "Scuola Normale Superiore":        "SNS",
}

def _acronym(label: str) -> str:
    return INSTITUTION_ACRONYMS.get(
        label, "".join(w[0] for w in label.split() if w[0].isalpha()).upper()
    )


def buttons_html(vis_map: dict, div_id: str, recip_top: int,
                 default: str | None = None) -> str:
    labels = list(vis_map.keys())
    if default is None:
        default = labels[0]
    btns = "\n".join(
        f'<button class="recip-btn{" active" if lab == default else ""}" '
        f'data-key="{lab}">{_acronym(lab)}</button>'
        for lab in labels
    )
    return f"""
{BUTTONS_CSS}
<div class="recip-toolbar" id="{div_id}-toolbar">
  <span class="recip-label">View by institution</span>
  {btns}
</div>
<script>
(function() {{
  var visMap  = {json.dumps(vis_map)};
  var divId   = "{div_id}";
  var toolbar = document.getElementById(divId + "-toolbar");
  toolbar.addEventListener("click", function(e) {{
    var btn = e.target.closest(".recip-btn");
    if (!btn) return;
    var key = btn.dataset.key;
    toolbar.querySelectorAll(".recip-btn").forEach(function(b) {{
      b.classList.toggle("active", b === btn);
    }});
  }});
}})();
</script>
"""


# Notebook: figure with dropdown
fig, vis_map = build_reciprocity_figure(ALL_DATASETS, RECIP_TOP)   # defaults to first institution
fig.show()

#  HTML export: strip the dropdown, add pill buttons instead 
DIV_ID = "reciprocity-plot"

export_fig = go.Figure(fig)            # copy so the notebook figure keeps its dropdown
export_fig.layout.updatemenus = []     # remove dropdown from the web version

FONT_CSS = """
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600&family=Playfair+Display:wght@600;700&display=swap" rel="stylesheet">
"""

fragment = export_fig.to_html(full_html=False, include_plotlyjs="cdn",
                              div_id=DIV_ID, config={"responsive": True})

html = (FONT_CSS + buttons_html(vis_map, DIV_ID, RECIP_TOP)   # active pill = first institution
        + fragment)

try:
    BASE = Path(__file__).resolve().parent.parent
except NameError:
    BASE = Path.cwd()

OUT_DIR = BASE / ".." / ".." / "website" / "visualizations"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / "reciprocity.html"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    f.write(html)
print(f"Wrote {OUT_PATH}")

#### Reciprocity Scatter — Findings: Including Italian Partners

| Institution | Bilateral partners | Near-diagonal (±0.1) | Above diagonal | Below diagonal |
|---|---|---|---|---|
| UNIBO  | 458 | **68.1%** | 28.4% | 3.5% |
| UNIMI  | 455 | **71.2%** | 12.6% | 16.3% |
| UNIPD  | 467 | **73.8%** | 17.9% | 8.3% |
| UNITO  | 462 | **61.0%** | 33.5% | 5.5% |
| UPO    | 461 | **66.7%** | 27.4% | 5.9% |
| SNS    | 458 | **77.1%** | 15.9% | 7.0% |

**Near-diagonal shares are the highest of the two modes across all six institutions.** Italian partner institutions overwhelmingly cluster near the diagonal — domestic inter-institutional citation exchange is symmetric — so including them raises the near-diagonal proportion everywhere, most visibly at SNS (77.1%) and UNIPD (73.8%).

**Italian institutions appear as mid-sized symmetric bubbles in every scatter.** Sapienza, University of Padua, and University of Bologna sit close to the dashed diagonal line across all institution scatters. Domestic relationships add citation volume without adding asymmetry.

**UNIMI's below-diagonal cluster is driven by two overlapping components in this mode.** Chinese biomedical institutions form the international component; Italian clinical research networks (IRCCS, CNR) form the domestic component. Together they account for the 16.3% below-diagonal share. IRCCS appears as a substantial bubble below the diagonal unique to UNIMI's scatter.

**UNITO is the most asymmetric institution even with Italian partners included** (near-diagonal 61.0%, above-diagonal 33.5%). The Italian symmetric buffer does not compensate for UNITO's strong outgoing dependency on US and Swiss physics institutions.

**SNS achieves the highest near-diagonal share (77.1%)**, with domestic partners (University of Pisa, University of Bologna, Sapienza, University of Padua) clustering tightly near the diagonal. The above-diagonal astrophysics infrastructure cluster (CfA, Space Telescope, Goddard) is present but small relative to the symmetric core.


#### Reciprocity Scatter — Findings: Excluding Italian Partners

| Institution | Bilateral partners | Near-diagonal (±0.1) | Above diagonal | Below diagonal |
|---|---|---|---|---|
| UNIBO  | 442 | 64.7% | 31.6% | 3.7% |
| UNIMI  | 442 | 69.0% | 13.8% | **17.2%** |
| UNIPD  | 455 | 71.4% | 19.3% | 9.2% |
| UNITO  | 450 | 58.4% | **36.2%** | 5.3% |
| UPO    | 455 | 64.0% | 29.9% | 6.2% |
| SNS    | 460 | 73.5% | 18.3% | 8.3% |

**All asymmetries are more pronounced without Italian partners.** Near-diagonal shares drop uniformly across all six institutions and above-diagonal shares rise, because the symmetric Italian cluster is replaced by international institutions that more often sit above the diagonal.

**UNITO's outgoing skew is the most extreme in this mode** (above-diagonal 36.2%, near-diagonal 58.4%). Without the buffering effect of near-symmetric Italian partners, UNITO's outgoing citation dependency on US and Swiss physics institutions becomes the dominant structural feature of its scatter, and it stands out as the most asymmetric institution in the dataset.

**UNIMI's below-diagonal share is highest in this mode (17.2%).** With Italian clinical networks removed, the Chinese biomedical cluster becomes proportionally larger and more visible. The below-diagonal concentration is driven entirely by Chinese and Southern European medical institutions — a cleaner and more interpretable signal than in the inclusion mode.

**SNS remains the most symmetric institution (73.5% near-diagonal)** and its above-diagonal astrophysics cluster is unchanged. The loss of Italian partners does not structurally alter SNS's scatter, confirming that SNS's symmetry is driven by its international physics community, not its domestic ties.

**This mode is directly comparable to the country-level notebook.** The scope, exclusion logic, and partner universe are equivalent, making cross-notebook comparisons of asymmetry distributions valid.


#### Reciprocity Scatter — Comparing the Two Perspectives

**What is stable across both modes:** The rank order of institutions by near-diagonal share is preserved (SNS highest, UNITO lowest in both modes). CNRS is the largest symmetric bubble in both. The Chinese biomedical below-diagonal cluster at UNIMI is present in both, as is the astrophysics above-diagonal cluster at SNS. UNITO is the most asymmetric institution in both modes.

**What changes:** Including Italian partners raises near-diagonal shares by 3–5 percentage points across all six institutions. The effect is largest at SNS (+3.6 pp) and UNIPD (+2.4 pp), where domestic partners are numerous and symmetric. UNIMI's below-diagonal cluster partially changes composition: the IRCCS domestic component is visible only in the inclusion mode, making the below-diagonal signal more heterogeneous there than in the exclusion mode.

**Which mode to use:** The exclusion mode gives cleaner separation between above- and below-diagonal signals (international asymmetry without domestic noise) and enables direct comparison with the country-level analysis. The inclusion mode is preferred when the goal is to capture the full bilateral relationship network, including the domestic inter-institutional layer.


---

## 3. Cross-Institution Comparison

### 3a — Proportional Partner Mix

Each bar shows the top-10 partner organisations as a share of that institution's total citation volume (incoming + outgoing combined), stacked to 100%. Comparing bar composition across institutions reveals which partnerships are disproportionately large for a specific institution relative to the others.


In [ ]:
def plot_proportional_stacked(datasets: dict, top_n: int = 10) -> go.Figure:
    rows = []
    for inst, (inb, out) in datasets.items():
        inb_agg = inb.groupby(["legal_name","country_name"])["count"].sum().reset_index().rename(columns={"count":"incoming"})
        out_agg = out.groupby(["legal_name","country_name"])["count"].sum().reset_index().rename(columns={"count":"outgoing"})
        m = pd.merge(inb_agg, out_agg, on=["legal_name","country_name"], how="outer").fillna(0)
        m["total"] = m["incoming"] + m["outgoing"]
        m["share"] = m["total"] / m["total"].sum() * 100
        for _, row in m.nlargest(top_n, "total").iterrows():
            rows.append({"institution": INSTITUTION_LABELS[inst],
                         "legal_name": row["legal_name"],
                         "country_name": row["country_name"],
                         "share": row["share"], "total": row["total"]})

    df  = pd.DataFrame(rows)
    fig = px.bar(df, x="institution", y="share", color="legal_name", text="legal_name",
                 hover_data={"total":":,","country_name":True,"share":":.2f"},
                 title=f"Top-{top_n} External Partners as % of Total Citation Volume",
                 labels={"share":"Share of total citations (%)","institution":""})
    fig.update_traces(textposition="inside", textfont_size=7, insidetextanchor="middle")
    fig.update_layout(template="plotly_white", height=540, title_x=0.5,
                      showlegend=False, bargap=0.25,
                      yaxis_title="Share of total citations (%)")
    return fig

plot_proportional_stacked(ALL_DATASETS, top_n=10).show()


#### Proportional Partner Mix — Findings: Including Italian Partners

**CNRS dominates the base of every bar, but its share varies meaningfully across institutions.** It accounts for a noticeably larger slice of SNS's total volume than of UNIBO or UNIMI's, reflecting that smaller and more specialised institutions have more concentrated partner portfolios.

**SNS shows the most concentrated partner structure overall**, with its top-10 partners accounting for the largest combined share of total citations across all six institutions.

**Italian institutions claim substantial portions at UPO and UNIMI.** At UPO, domestic partners account for roughly half the combined top-10 share, making its bar the most distinctive in the chart. At UNIMI, IRCCS hospital networks occupy the third-largest slice — a slot filled by international research universities at every other institution.

**This mode best answers:** what does each institution's full citation neighbourhood look like, including domestic ties?

#### Proportional Partner Mix — Findings: Excluding Italian Partners

**CNRS's share becomes proportionally larger at every institution** once Italian partners are removed from the top-10 ranking. The effect is most visible at SNS, where CNRS now occupies the widest base slice in the chart.

**UNIMI's international mix is more similar to UNIPD's than it appears in the inclusion mode.** Without IRCCS and CNR claiming domestic slots, UNIMI's bar reveals an international partner composition — US universities, French research councils, UK institutions — that closely parallels UNIPD's structure. The two institutions appear more alike in this mode than in any other chart in the notebook.

**UPO's top-10 shifts from domestically concentrated to internationally oriented.** With seven Italian partners removed, UPO's list is now led by physics infrastructure (CERN, IN2P3) and French and US research universities. While UPO and SNS share some partners in this mode (CERN, Université Paris-Saclay, CSIC), SNS remains more concentrated in US elite universities (Caltech, MIT), so the two profiles converge in character but not in composition.

**This mode best answers:** how do the six institutions compare in their international citation reach, and how does this compare to the country-level analysis?

#### Proportional Partner Mix — Comparing the Two Perspectives

**What is stable across both modes:** CNRS's position at the base of every bar, SNS showing the most concentrated partner structure in both modes, and Harvard University Press appearing as a visible slice at five of six institutions.

**What changes:** The most dramatic shift is at UPO, where the entire composition of the bar changes — from half domestic to a physics-infrastructure-led international mix. At UNIMI, the exclusion of IRCCS and CNR makes its international bar converge toward UNIPD's, revealing that UNIMI's distinctive domestic embeddedness, not its international profile, sets it apart from the other institutions.

**Which mode to use:** The inclusion mode reveals the full partner ecosystem and is the appropriate reference when studying Italian academic inter-institutional networks. The exclusion mode is appropriate for comparing international citation concentration across the six institutions on a common basis, and for cross-referencing against the country-level proportional stacked bar.

## 4. Summary of Findings

### Key institutional patterns (stable across both modes)

**1. CNRS as the universal anchor.** CNRS is the top partner by citation volume at all six institutions and maintains near-symmetric citation exchange at each one in both modes. It is the only organisation that is simultaneously prominent and balanced across the entire dataset regardless of the Italian partner filter.

**2. Each institution has a distinct citation balance — asymmetries are more pronounced in the exclusion mode.** UNIMI attracts the most incoming citations relative to outgoing (below-diagonal share: 16.3% with Italy, 17.2% without), driven by Italian and Chinese medical institutions in the inclusion mode, and by Chinese and Southern European institutions alone in the exclusion mode. UNITO generates the most outgoing citations relative to incoming (above-diagonal share: 33.5% with Italy, 36.2% without) — its outgoing skew is structural and holds in both modes, but is more visible without the buffering effect of near-symmetric Italian partners. SNS achieves the most symmetric exchange overall (77.1% near-diagonal with Italy, 73.5% without), and is the only institution where CERN — not Harvard University Press — is the dominant above-diagonal partner in both modes.

**3. Harvard University Press is the dominant outgoing sink at five of six institutions** — absent only at SNS. Its asymmetry is highest at UPO (+0.31), the most specialised institution. This finding holds in both modes; its visual prominence is slightly stronger in the exclusion mode, where Italian symmetric partners no longer dilute the effect.

**4. Incoming diversity exceeds outgoing diversity at every institution.** Italian universities are cited by a broader range of partner organisations than they actively cite — a structural feature that holds regardless of institution size, disciplinary focus, or Italian partner filter setting.

---

### The domestic layer: what the inclusion mode adds

The Italian partner filter operationalises the boundary between two analytical layers that the country-level analysis cannot separate. The exclusion mode aligns with the country-level analysis and isolates international citation flows, enabling direct cross-notebook comparison. The inclusion mode adds a layer invisible at the country level: the domestic inter-institutional citation network within Italy.

This domestic layer is not uniform. **UNIBO and UNIPD** maintain a strong bilateral tie — each ranks in the other's top-15 with broadly symmetric bars, the most balanced large Italian–Italian pair in the dataset. **UNIMI** is uniquely embedded in Italy's clinical research network: IRCCS hospital institutions and CNR cite UNIMI substantially more than UNIMI cites them back, making UNIMI a primary reference institution for Italian applied and medical research. **UPO** is the most domestically concentrated institution overall, with seven of its fifteen top partners being Italian — a signal that disappears entirely in the exclusion mode, where UPO's profile converges toward SNS.

Together, the two modes show that Italian academic institutions operate simultaneously within an international citation ecosystem and a dense domestic one — and that the relative weight of these two layers varies substantially by institution. For multidisciplinary institutions like UNIBO and UNIPD, the two layers are roughly comparable in scale. For UNIMI, the domestic clinical layer is structurally distinctive. For UPO, it is dominant. For SNS and UNITO, whose citation networks are shaped primarily by international physics and computational research communities, the domestic layer is present but secondary.